Prepare intent data

In [1]:
import os
import pandas as pd
from datasets import load_dataset
from sklearn.model_selection import train_test_split

def prepare_intent_dataset():
    # 1. Ensure data storage directory exists
    os.makedirs("data", exist_ok=True)
    print("📁 'data/' directory verified or created.")

    # 2. Fetch the Bitext dataset from Hugging Face
    print("📥 Downloading Bitext customer support dataset...")
    raw_dataset = load_dataset("bitext/Bitext-customer-support-llm-chatbot-training-dataset")

    # Convert the training split into a pandas DataFrame
    df = pd.DataFrame(raw_dataset['train'])
    print(f"✅ Loaded {len(df)} raw rows.")

    # 3. Explicit mapping of the 27 intents into Binary Classes
    # Label 0: Informative (FAQs, Lookups, Policies)
    # Label 1: Actionable (Requires transactional operations/state mutation)
    intent_mapping = {
        # --- INFORMATIVE / RETRIEVAL PATH (Label 0) ---
        "check_cancellation_fee": 0,
        "check_invoices": 0,
        "check_payment_methods": 0,
        "check_refund_policy": 0,
        "delivery_options": 0,
        "delivery_period": 0,
        "get_invoice": 0,
        "newsletter_subscription": 0,
        "registration_problems": 0,
        "review": 0,
        "complaint": 0,
        "contact_customer_service": 0,
        "contact_human_agent": 0,
        "track_order": 0,
        "track_refund": 0,

        # --- ACTIONABLE / LLM GENERATION PATH (Label 1) ---
        "cancel_order": 1,
        "change_order": 1,
        "change_shipping_address": 1,
        "set_up_shipping_address": 1,
        "place_order": 1,
        "get_refund": 1,
        "edit_account": 1,
        "create_account": 1,
        "delete_account": 1,
        "switch_account": 1,
        "recover_password": 1,
        "payment_issue": 1
    }

    # 4. Filter out columns and create labels
    # Use 'instruction' as the feature input and 'intent' to compute our binary label
    df['label'] = df['intent'].map(intent_mapping)

    # Keep only the essential data columns for intent routing
    processed_df = df[['instruction', 'intent', 'label']].rename(columns={'instruction': 'text'})

    # Drop any unmapped rows just in case
    processed_df = processed_df.dropna(subset=['label'])
    processed_df['label'] = processed_df['label'].astype(int)

    # 5. Diagnostic: print out dataset balance
    print("\n📊 Dataset Distribution Check:")
    print(processed_df['label'].value_counts(normalize=True))
    # show the full dataset
    print(processed_df)
    # 6. Perform a Stratified Train/Test Split (80/20)
    # Stratifying ensures both splits contain identical proportions of Class 0 and Class 1
    train_df, test_df = train_test_split(
        processed_df,
        test_size=0.2,
        random_state=42,
        stratify=processed_df['label']
    )

    # 7. Save outputs to disk
    train_path = os.path.join("data", "intent_train.csv")
    test_path = os.path.join("data", "intent_test.csv")

    train_df.to_csv(train_path, index=False)
    test_df.to_csv(test_path, index=False)

    print("\n🎉 Data preprocessing complete!")
    print(f"💾 Saved Training Split ({len(train_df)} rows) ➡️ {train_path}")
    print(f"💾 Saved Testing Split ({len(test_df)} rows) ➡️ {test_path}")

if __name__ == "__main__":
    prepare_intent_dataset()

📁 'data/' directory verified or created.
📥 Downloading Bitext customer support dataset...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:124: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/11.9k [00:00<?, ?B/s]

Bitext_Sample_Customer_Support_Training_(…):   0%|          | 0.00/19.2M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/26872 [00:00<?, ? examples/s]

✅ Loaded 26872 raw rows.

📊 Dataset Distribution Check:
label
0    0.538265
1    0.461735
Name: proportion, dtype: float64
                                                    text        intent  label
0       question about cancelling order {{Order Number}}  cancel_order      1
1      i have a question about cancelling oorder {{Or...  cancel_order      1
2        i need help cancelling puchase {{Order Number}}  cancel_order      1
3             I need to cancel purchase {{Order Number}}  cancel_order      1
4      I cannot afford this order, cancel purchase {{...  cancel_order      1
...                                                  ...           ...    ...
26867  I am waiting for a rebate of {{Refund Amount}}...  track_refund      0
26868  how to see if there is anything wrong with my ...  track_refund      0
26869  I'm waiting for a reimbjrsement of {{Currency ...  track_refund      0
26870  I don't know what to do to see my reimbursemen...  track_refund      0
26871  I need to kn

Fine Tuning starts

In [2]:
# ==========================================
# 1. INSTALL REQUIRED LIBRARIES
# ==========================================
# We need transformers and accelerate for training, datasets for data handling,
# and sentencepiece because DeBERTa-v3 relies on it for tokenization.
!pip install -q transformers datasets accelerate evaluate scikit-learn sentencepiece

import os
import torch
import numpy as np
import pandas as pd
from datasets import Dataset, DatasetDict
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding,
    pipeline
)
import evaluate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 6.5 MB/s eta 0:00:00


In [3]:
# ==========================================
# 2. LOAD AND PREPARE DATASETS
# ==========================================
print("🔄 Loading CSV datasets into Hugging Face Dataset format...")
# Load the CSV files we uploaded into Pandas DataFrames
train_df = pd.read_csv("data/intent_train.csv")
test_df = pd.read_csv("data/intent_test.csv")

# DeBERTa expects the text column to be named 'text' and label to be named 'label'
# (Our previous script already formatted them as 'text' and 'label')
train_dataset = Dataset.from_pandas(train_df)
test_dataset = Dataset.from_pandas(test_df)

# Combine them into a single DatasetDict for clean mapping
raw_datasets = DatasetDict({
    "train": train_dataset,
    "test": test_dataset
})

🔄 Loading CSV datasets into Hugging Face Dataset format...


In [4]:
# ==========================================
# 3. TOKENIZATION
# ==========================================
MODEL_CKPT = "distilbert-base-uncased"
print(f"📥 Loading tokenizer for {MODEL_CKPT}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_CKPT, use_fast=True)

def tokenize_function(examples):
    # Truncate queries longer than 128 tokens. Customer support inputs are usually short,
    # so keeping max_length small saves a massive amount of GPU memory and training time.
    return tokenizer(examples["text"], truncation=True, max_length=128)

print("⚡ Tokenizing datasets...")
tokenized_datasets = raw_datasets.map(tokenize_function, batched=True)

# Data collator dynamically pads the batches to the maximum length in that specific batch,
# which is much faster than padding everything to a static 128 length.
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

📥 Loading tokenizer for distilbert-base-uncased...


config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

⚡ Tokenizing datasets...


Map:   0%|          | 0/20697 [00:00<?, ? examples/s]

Map:   0%|          | 0/5175 [00:00<?, ? examples/s]

In [5]:
# ==========================================
# 4. DEFINE EVALUATION METRICS
# ==========================================
# Load accuracy and f1 metrics to monitor performance during training epochs
accuracy_metric = evaluate.load("accuracy")
f1_metric = evaluate.load("f1")

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    # Convert model logits (raw outputs) into class predictions (0 or 1) via argmax
    preds = np.argmax(predictions, axis=1)

    acc = accuracy_metric.compute(predictions=preds, references=labels)["accuracy"]
    f1 = f1_metric.compute(predictions=preds, references=labels, average="binary")["f1"]
    return {"accuracy": acc, "f1": f1}

In [6]:
# ==========================================
# 5. INITIALIZE THE MODEL
# ==========================================
print(f"🤖 Initializing model: {MODEL_CKPT} for Binary Classification...")
# num_labels=2 sets up the sequence classification head with 2 output nodes
model = AutoModelForSequenceClassification.from_pretrained(MODEL_CKPT, num_labels=2)

🤖 Initializing model: distilbert-base-uncased for Binary Classification...


model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [7]:
# ==========================================
# 6. CONFIGURING TRAINING ARGUMENTS (STABILIZED)
# ==========================================
training_args = TrainingArguments(
    output_dir="./deberta-intent-router",      # Directory where checkpoints are saved
    learning_rate=2e-5,                        # Standard stable learning rate for DeBERTa
    per_device_train_batch_size=16,            # Fits comfortably inside T4 memory
    per_device_eval_batch_size=16,
    num_train_epochs=3,                        # 3 epochs is ideal
    weight_decay=0.01,                         # Regularization technique to prevent overfitting
    eval_strategy="epoch",               # Evaluate loss/metrics at the end of every epoch
    save_strategy="epoch",                     # Save a checkpoint at the end of every epoch
    load_best_model_at_end=True,               # Keep the checkpoint that performed best
    metric_for_best_model="f1",                # Optimize for the highest F1-score
    fp16=True,                                # 🔴 CHANGED TO FALSE: Solves the DeBERTa NaN overflow bug
    max_grad_norm=1.0,                         # 🟢 ADDED: Prevents gradient explosion by clipping them at 1.0
    report_to="none"                           # Disable third-party logging
)

# Pass everything into the Hugging Face Trainer API
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["test"],
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

In [8]:
# ==========================================
# 7. EXECUTE FINE-TUNING
# ==========================================
print("🚀 Starting fine-tuning loop...")
trainer.train()

print("\n🎯 Evaluating final model on test split...")
evaluation_results = trainer.evaluate()
print(f"Final Test Accuracy: {evaluation_results['eval_accuracy']:.4f}")
print(f"Final Test F1-Score: {evaluation_results['eval_f1']:.4f}")

🚀 Starting fine-tuning loop...


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.019260,0.009644,0.998261,0.998118
2,0.002424,0.007634,0.998647,0.998533
3,0.001308,0.010640,0.998841,0.998745


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


🎯 Evaluating final model on test split...


Training Loss,Validation Loss,Epoch,Accuracy,F1
0.001308,0.010640,3,0.998841,0.998745


Final Test Accuracy: 0.9988
Final Test F1-Score: 0.9987


In [11]:
# ==========================================
# 8. SAVE THE FINE-TUNED WEIGHTS
# ==========================================
# This saves the model weights, configuration, and tokenizer vocabulary locally in Colab
final_model_path = "./fine_tuned_distilbert_router"
model.save_pretrained(final_model_path)
tokenizer.save_pretrained(final_model_path)
print(f"💾 Model and tokenizer successfully saved to: {final_model_path}")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

💾 Model and tokenizer successfully saved to: ./fine_tuned_distilbert_router


In [12]:
# ==========================================
# 9. LOCAL PIPELINE TESTING (VERIFICATION)
# ==========================================
print("\n🔍 Verification testing on sample prompts:")
# Create an end-to-end classification pipeline using our trained model
classifier = pipeline("text-classification", model=final_model_path, tokenizer=final_model_path, device=0)

test_prompts = [
    "Where is my package?",            # Expected: LABEL_0 (Informative)
    "Cancel my package right now"      # Expected: LABEL_1 (Actionable)
]

for prompt in test_prompts:
    result = classifier(prompt)[0]
    print(f"Prompt: '{prompt}' ➡️ Prediction: {result['label']} (Confidence: {result['score']:.4f})")


🔍 Verification testing on sample prompts:


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

Prompt: 'Where is my package?' ➡️ Prediction: LABEL_0 (Confidence: 0.9828)
Prompt: 'Cancel my package right now' ➡️ Prediction: LABEL_1 (Confidence: 1.0000)


## Push Model to Hugging Face Hub

In [15]:
from huggingface_hub import notebook_login
notebook_login() # Paste your Hugging Face write token here

In [16]:
# Define Hugging Face repository ID
hf_repo_id = "mahdi2020/fine-tuned-distilbert-customer-intent-router"

# Push the model and tokenizer to the Hugging Face Hub
model.push_to_hub(hf_repo_id)
tokenizer.push_to_hub(hf_repo_id)

print(f"🎉 Model and tokenizer successfully pushed to: https://huggingface.co/{hf_repo_id}")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...71w1jpm/model.safetensors:   0%|          |  575kB /  268MB            

README.md:   0%|          | 0.00/5.17k [00:00<?, ?B/s]

🎉 Model and tokenizer successfully pushed to: https://huggingface.co/mahdi2020/fine-tuned-distilbert-customer-intent-router


Save the model as ZIP

In [13]:
import shutil
from google.colab import files

# Zip the directory
shutil.make_archive("fine_tuned_distilbert_router", 'zip', "fine_tuned_distilbert_router")

# Download it
files.download("fine_tuned_distilbert_router.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>